This Notebook is meant as an introduction to Pytorch using Logistic Regression as an example.

In [9]:
# Prepare some data from the breast cancer study

import torch
import torch.nn as nn

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#from math import exp
import matplotlib.pyplot as plt
df = pd.read_csv('wisconsonbc.csv')
df.columns

Index(['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean',
       'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave points_worst',
       'symmetry_worst', 'fractal_dimension_worst', 'Unnamed: 32'],
      dtype='str')

In [10]:
df = df[['radius_mean', 'texture_mean', 'diagnosis']]
df = pd.get_dummies(df)
df.head()

,radius_mean,texture_mean,diagnosis_B,diagnosis_M
0,17.99,10.38,False,True
1,20.57,17.77,False,True
2,19.69,21.25,False,True
3,11.42,20.38,False,True
4,20.29,14.34,False,True


In [11]:
# Next, extract a subset of columns to use for learning.
X = df[['radius_mean', 'texture_mean']]
y = df['diagnosis_M']

In [12]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features for better training performance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)


In [15]:
# sanity check
X_train_tensor[:10]

tensor([[-1.4408, -0.4353],
        [ 1.9741,  1.7330],
        [-1.4000, -1.2496],
        [-0.9818,  1.4162],
        [-1.1177, -1.0103],
        [ 0.1196,  1.9607],
        [ 0.0828,  0.1279],
        [-0.7610, -0.8906],
        [-0.5288, -0.2922],
        [ 1.6343,  0.2523]])

We now have a set of data to use for training.  Let's proceed to define the logistic model structure.

In [21]:
# Logistic Regression is a linear model under the covers
class LogisticRegression(nn.Module):
    def __init__(self, input_size): # instantiate the model object
        super(LogisticRegression, self).__init__()
        self.linear = nn.Linear(input_size, 1)

    def forward(self, x): # Run model inference
        return torch.sigmoid(self.linear(x))

Now we can check the model to make sure everything runs.  we will run a single inference as a sanity check.

In [26]:
# Instantiate the model
model_inf = LogisticRegression(X_train_tensor.shape[1]) #Pass in the vector dim of one input

with torch.no_grad():  # do not calculate gradients during inference
    y_hat = model_inf(X_train_tensor[0,:])
    
print(y_hat)


tensor([0.5104])


Now we write a training loop.  This is the most complex part of the process.  we need several objects: 

 - A model instantiated for training.
 - An optimization method
 - A loss function to minimize
 - Number of times to pass trough the training data (epochs)
 - The number of inputs to present to the model on each iteration

We then use these objects in a training loop to train the model weights.

In [32]:
# instantiate objects for training.
# Initialize the model
in_size = X_train_tensor.shape[1]
model = LogisticRegression(in_size)

# Print model architecture
print(model)

# Define the binary cross-entropy loss function
loss_fn = nn.BCELoss()

# Define the optimizer (Stochastic Gradient Descent)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# Set the number of epochs and batch size
epochs = 1000
batch_size = 32
n_batches = X_train_tensor.shape[0] // batch_size



LogisticRegression(
  (linear): Linear(in_features=2, out_features=1, bias=True)
)


In [34]:
# The training loop
# Training loop

for epoch in range(epochs):
    for i in range(n_batches):
        # Get batch of data
        start = i * batch_size
        end = start + batch_size
        X_batch = X_train_tensor[start:end]
        y_batch = y_train_tensor[start:end]

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass: Compute predicted y by passing X_batch to the model
        y_pred = model(X_batch)

        # Compute the loss
        loss = loss_fn(y_pred, y_batch)

        # Backward pass: Compute gradient of the loss with respect to model parameters
        loss.backward()

        # Update weights
        optimizer.step()

    # Print loss every 100 epochs
    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')



/home/joe/Documents/Gonzaga/2026-F/DATA 575/DATA575/.venv/lib/python3.13/site-packages/torch/autograd/graph.py:1072: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch [100/1000], Loss: 0.3040
Epoch [200/1000], Loss: 0.2767
Epoch [300/1000], Loss: 0.2696
Epoch [400/1000], Loss: 0.2671
Epoch [500/1000], Loss: 0.2660
Epoch [600/1000], Loss: 0.2656
Epoch [700/1000], Loss: 0.2655
Epoch [800/1000], Loss: 0.2654
Epoch [900/1000], Loss: 0.2654
Epoch [1000/1000], Loss: 0.2655


We have trained our logistic model.  Now we want to know how well (or poorly) it works.

In [35]:
from sklearn.metrics import accuracy_score

# Put the model in evaluation mode
model.eval()

# Predict probabilities on the test set
with torch.no_grad():  # No need to compute gradients during evaluation
    y_pred_probs = model(X_test_tensor)
    y_pred = (y_pred_probs >= 0.5).float()  # Convert probabilities to binary outputs

# Convert tensors to numpy arrays
y_pred_np = y_pred.numpy()
y_test_np = y_test_tensor.numpy()

# Calculate accuracy
accuracy = accuracy_score(y_test_np, y_pred_np)
print(f"Test Accuracy: {accuracy * 100:.2f}%")


Test Accuracy: 90.35%


In [40]:
from sklearn.metrics import confusion_matrix

c = confusion_matrix(y_test_np, y_pred_np)
print(c)

[[67  4]
 [ 7 36]]


In [42]:
# precision = tp / (tp + fp) 
precision = c[0,0] / c[0,0] + c[1,0]

# recall = tp / (tp + fn)
recall = c[0,0] / c[0,0] + c[0,1]

print(f'precision = {precision}')
print(f'recall = {recall}')


precision = 8.0
recall = 5.0
